# Final Testing — Multisource AI Content Intelligence Platform

This notebook performs final end-to-end testing of the AI Content Intelligence Platform.

The complete pipeline includes:

- Content ingestion
- Text cleaning
- Text chunking
- Embeddings
- Vector storage
- Semantic retrieval
- RAG-based question answering
- LLM-powered responses

The goal is to verify that the complete pipeline works correctly from
content ingestion to final question answering.

In [57]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [58]:
import os

from dotenv import load_dotenv

from langchain_groq import ChatGroq

from src.ingestion.pdf_loader import load_pdf
from src.processing.text_processor import split_documents
from src.embeddings.embedding_service import get_embedding_model
from src.retrieval.vector_store import create_vector_store
from src.retrieval.retriever import create_retriever
from src.rag.qa_chain import answer_question

## 1. Initialize Environment and LLM

Load the API key from the environment and initialize the Groq LLM
used for RAG-based question answering.

In [59]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY was not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=groq_api_key,
    temperature=0
)


## 2. Initialize Embedding Model

Initialize the sentence-transformer embedding model used to convert
document chunks and user queries into vector representations.

In [60]:
embedding_model = get_embedding_model()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2700.78it/s]


## 3. Load Test Content

Load a PDF document as the knowledge source for the final RAG test.

In [33]:
pdf_path = r"C:\Users\maury\Downloads\Linear Regression.pdf"

documents = load_pdf(pdf_path)

print("PDF loaded successfully.")
print("Number of pages:", len(documents))

PDF loaded successfully.
Number of pages: 5


## 4. Process and Chunk Content

Clean the extracted text and split the document into smaller chunks
for embedding and semantic retrieval.

In [34]:
chunks = split_documents(documents)

print("Document processing completed.")
print("Number of chunks:", len(chunks))

Document processing completed.
Number of chunks: 6


In [35]:
print("First chunk:")
print(chunks[0].page_content[:500])

print("\nMetadata:")
print(chunks[0].metadata)

First chunk:
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervised ML algorithm for regression problems. LR models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyperplane) to the data. In the simplest form the linear regression can be understood by taking an example of single input feature (x) & output (y). LR is a regression algorithm that tries to predi ct th

Metadata:
{'producer': 'pdfcpu v0.15.0 dev', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-26T17:34:59+00:00', 'author': 'Studio', 'moddate': '2026-09-26T17:34:59+00:00', 'title': 'Linear Regression (Notes)', 'source': 'C:\\Users\\maury\\Downloads\\Linear Regression.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


## 5. Create Vector Store

Convert the processed document chunks into embeddings and store them
in a Chroma vector store for semantic search.

In [36]:
vector_store = create_vector_store(
    chunks,
    embedding_model
)


## 6. Create Retriever

Create a retriever that searches the vector store for the most relevant
document chunks for a given question.

In [37]:
retriever = create_retriever(
    vector_store,
    k=3
)


## 7. Test Semantic Retrieval

Test whether the system can retrieve relevant information from the
document before passing the retrieved context to the LLM.

In [38]:
query = "What is linear regression?"

retrieved_documents = retriever.invoke(query)

print("Number of retrieved documents:", len(retrieved_documents))

for i, document in enumerate(retrieved_documents):
    print(f"\n--- Retrieved Document {i + 1} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)

Number of retrieved documents: 3

--- Retrieved Document 1 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervised ML algorithm for regression problems. LR models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyperplane) to the data. In the simplest form the linear regression can be understood by taking an example of single input feature (x) & output (y). LR is a regression algorithm that tries to predi ct th
Metadata: {'author': 'Studio', 'title': 'Linear Regression (Notes)', 'total_pages': 5, 'producer': 'pdfcpu v0.15.0 dev', 'creationdate': '2026-09-26T17:34:59+00:00', 'source': 'C:\\Users\\maury\\Downloads\\Linear Regression.pdf', 'page_label': '1', 'moddate': '2026-09-26T17:34:59+00:00', 'creator': 'Microsoft® Word LTSC', 'page': 0}

--- Retrieved Document 2 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Li

## 8. RAG-Based Question Answering

Use the retriever to obtain relevant context and the LLM to generate
an answer based only on the retrieved content.

In [39]:
question = "What is linear regression?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

Question:
What is linear regression?

Answer:
Linear regression is a supervised machine‑learning algorithm used for regression problems. It models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyperplane) to the data. In its simplest form, with a single input feature \(x\) and output \(y\), the model seeks the best‑fit line described by the equation \(y = mx + c\). For multiple independent features, it becomes a multiple linear regression, where the goal is to estimate all parameters \(\theta_i\) so that the resulting line or hyperplane best predicts \(y\).


## 9. Retrieved Sources

Display the document chunks used by the RAG pipeline to generate the
answer.

In [40]:
print("Sources used for the answer:")

for i, document in enumerate(result["sources"]):
    print(f"\n--- Source {i + 1} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)

Sources used for the answer:

--- Source 1 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervised ML algorithm for regression problems. LR models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyperplane) to the data. In the simplest form the linear regression can be understood by taking an example of single input feature (x) & output (y). LR is a regression algorithm that tries to predi ct th
Metadata: {'source': 'C:\\Users\\maury\\Downloads\\Linear Regression.pdf', 'page': 0, 'creationdate': '2026-09-26T17:34:59+00:00', 'page_label': '1', 'author': 'Studio', 'moddate': '2026-09-26T17:34:59+00:00', 'creator': 'Microsoft® Word LTSC', 'producer': 'pdfcpu v0.15.0 dev', 'title': 'Linear Regression (Notes)', 'total_pages': 5}

--- Source 2 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervi

## 10. Additional Question Test

Test the RAG pipeline with another question related to the document.

In [41]:
question = "What is the purpose of the cost function in linear regression?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

Question:
What is the purpose of the cost function in linear regression?

Answer:
The cost function measures how far the predicted values are from the actual values. By minimizing this cost (e.g., using Mean Squared Error), we find the best‑fit line that gives the lowest prediction error.


## 11. Test Out-of-Context Question

Verify that the system does not invent an answer when the requested
information cannot be found in the provided content.

In [42]:
question = "What is the capital city of France?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

Question:
What is the capital city of France?

Answer:
I could not find the answer in the provided content.


## 12. Final End-to-End Test

The following test runs the complete pipeline:

PDF ingestion → text processing → chunking → embeddings →
vector storage → semantic retrieval → RAG → LLM answer.

In [43]:
final_question = "Explain the basic idea of linear regression."

final_result = answer_question(
    final_question,
    retriever,
    llm
)

print("=" * 60)
print("FINAL RAG TEST")
print("=" * 60)

print("\nQuestion:")
print(final_question)

print("\nAnswer:")
print(final_result["answer"])

print("\nNumber of sources retrieved:")
print(len(final_result["sources"]))

print("\nSource types:")

for document in final_result["sources"]:
    print(document.metadata.get("type", "unknown"))

print("\n" + "=" * 60)
print("END-TO-END TEST COMPLETED")
print("=" * 60)

FINAL RAG TEST

Question:
Explain the basic idea of linear regression.

Answer:
Linear regression is a supervised machine‑learning algorithm used for regression problems.  
It models the relationship between a dependent variable (the output) and one or more independent variables (the inputs) by fitting the best‑possible straight line (or, in higher dimensions, a plane or hyperplane) to the data.  

In its simplest form with a single input feature \(x\) and output \(y\), the model seeks the line  

\[
y = mx + c
\]

where \(m\) is the slope and \(c\) is the intercept.  
With multiple features, the model becomes

\[
y = \theta_0 + \theta_1x_1 + \theta_2x_2 + \dots + \theta_nx_n,
\]

where each \(\theta_i\) is a weight (coefficient) that the algorithm estimates so that the resulting line/plane best predicts \(y\).

Number of sources retrieved:
3

Source types:
unknown
unknown
unknown

END-TO-END TEST COMPLETED


The final test verifies the complete retrieval-augmented generation
pipeline.

The system successfully performs:

1. PDF content ingestion
2. Text cleaning and chunking
3. Embedding generation
4. Vector storage using Chroma
5. Semantic retrieval
6. Context-based question answering
7. Source retrieval
8. Out-of-context question handling

The RAG pipeline is now ready to be extended with additional content
intelligence capabilities such as summarization, key point extraction,
topic identification, and content insights.

## 13. Content Intelligence Analysis

Analyze the source content and generate a structured content
intelligence report containing a summary, key points, topics,
important concepts, and insights.

In [44]:
from src.intelligence.content_analyzer import analyze_content

In [45]:
analysis = analyze_content(
    documents,
    llm
)

print(analysis)

**1. Summary**  
The content explains linear regression as a supervised learning technique that models the relationship between one or more independent variables and a dependent variable by fitting a straight line (or hyperplane). It covers the hypothesis function, the cost function (Mean Squared Error), and the use of Gradient Descent to minimize this cost. It also mentions the Normal Equation as an analytical alternative, and lists the key assumptions underlying linear regression models.

---

**2. Key Points**  
- Linear regression predicts a continuous target by fitting a best‑fit line/plane.  
- Hypothesis function:  
  - Simple: \(y = mx + c\)  
  - Multiple: \(h_\theta(x) = \theta_0 + \theta_1x_1 + \dots + \theta_nx_n\)  
- Cost function (MSE):  
  \[
  J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}(h_\theta(x^{(i)})-y^{(i)})^2
  \]  
- Gradient Descent updates parameters:  
  \[
  \theta_k \leftarrow \theta_k - \alpha \frac{\partial J(\theta)}{\partial \theta_k}
  \]  
- Learning rate 

## 14. Combined Content Intelligence and RAG Test

Test the two core capabilities of the platform on the same source:

- Content Intelligence for structured content analysis
- RAG for context-grounded question answering

In [46]:
print("=" * 70)
print("MULTISOURCE AI CONTENT INTELLIGENCE — FINAL TEST")
print("=" * 70)

print("\n[1] CONTENT INTELLIGENCE")
print("-" * 70)
print(analysis)

print("\n[2] RAG QUESTION ANSWERING")
print("-" * 70)

final_question = "What is the role of gradient descent in linear regression?"

final_result = answer_question(
    final_question,
    retriever,
    llm
)

print("Question:", final_question)
print("\nAnswer:")
print(final_result["answer"])

print("\n[3] RETRIEVED SOURCES")
print("-" * 70)

for i, document in enumerate(final_result["sources"]):
    print(f"\nSource {i + 1}")
    print(document.page_content[:300])
    print("Metadata:", document.metadata)

print("\n" + "=" * 70)
print("FINAL TEST COMPLETED SUCCESSFULLY")
print("=" * 70)

MULTISOURCE AI CONTENT INTELLIGENCE — FINAL TEST

[1] CONTENT INTELLIGENCE
----------------------------------------------------------------------
**1. Summary**  
The content explains linear regression as a supervised learning technique that models the relationship between one or more independent variables and a dependent variable by fitting a straight line (or hyperplane). It covers the hypothesis function, the cost function (Mean Squared Error), and the use of Gradient Descent to minimize this cost. It also mentions the Normal Equation as an analytical alternative, and lists the key assumptions underlying linear regression models.

---

**2. Key Points**  
- Linear regression predicts a continuous target by fitting a best‑fit line/plane.  
- Hypothesis function:  
  - Simple: \(y = mx + c\)  
  - Multiple: \(h_\theta(x) = \theta_0 + \theta_1x_1 + \dots + \theta_nx_n\)  
- Cost function (MSE):  
  \[
  J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}(h_\theta(x^{(i)})-y^{(i)})^2
  \]  
- Gradie

## 15. Unified Content Loader Test

Test the unified ingestion interface that routes different content
sources to their corresponding source-specific loaders.

In [47]:
from src.ingestion.content_loader import load_content

In [48]:
test_pdf_path = r"C:\Users\maury\Downloads\Linear Regression.pdf"

loaded_documents = load_content(
    test_pdf_path,
    "pdf"
)
print("Source type: PDF")
print("Documents loaded:", len(loaded_documents))

Source type: PDF
Documents loaded: 5


In [49]:
youtube_url = "https://youtu.be/JJWE3QH-Q6s?list=RDB78jVe9mTCg"

youtube_documents = load_content(
    youtube_url,
    "youtube"
)

print("Source type: YouTube")
print("Documents loaded:", len(youtube_documents))

Source type: YouTube
Documents loaded: 1


In [50]:
try:
    load_content(
        "something",
        "mp4"
    )
except ValueError as e:
    print("Invalid source test passed.")
    print(e)

Invalid source test passed.
Unsupported source type: mp4. Supported types: pdf, docx, youtube, website.


In [51]:
from src.ingestion.content_loader import detect_source_type

In [52]:
pdf_path = r"C:\Users\maury\Downloads\Linear Regression.pdf"

print("PDF:", detect_source_type(pdf_path))

youtube_url = "https://youtu.be/JJWE3QH-Q6s"

print("YouTube:", detect_source_type(youtube_url))

docx_path = r"C:\Users\maury\Downloads\sample-document-small.docx"

print("DOCX:", detect_source_type(docx_path))

website_url = "https://example.com"

print("Website:", detect_source_type(website_url))

PDF: pdf
YouTube: youtube
DOCX: docx
Website: website


In [53]:
documents = load_content(pdf_path)

print("Automatically detected:", detect_source_type(pdf_path))
print("Documents loaded:", len(documents))

Automatically detected: pdf
Documents loaded: 5


In [54]:
try:
    detect_source_type("random_file.xyz")
except ValueError as e:
    print("Invalid source detection test passed.")
    print(e)

Invalid source detection test passed.
Could not determine the source type. Supported sources: PDF, DOCX, YouTube, and website URLs.


## 16. Persistent Vector Store Test

Verify that the Chroma vector store is stored locally and can be
used for semantic retrieval.

In [55]:
import os

vector_store_path = "data/vector_store"

print("Vector store directory exists:", os.path.exists(vector_store_path))

if os.path.exists(vector_store_path):
    print("Vector store contents:")
    print(os.listdir(vector_store_path))

Vector store directory exists: True
Vector store contents:
['1779d7e0-b393-4f0a-bfe5-9f44aeb05e3f', 'chroma.sqlite3']


In [63]:
query = "What is linear regression?"

persistent_results = vector_store.similarity_search(
    query,
    k=3
)

print("Retrieved documents:", len(persistent_results))

for i, document in enumerate(persistent_results):
    print(f"\n--- Result {i + 1} ---")
    print(document.page_content[:300])

Retrieved documents: 3

--- Result 1 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervised ML algorithm for regression problems. LR models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyp

--- Result 2 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervised ML algorithm for regression problems. LR models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyp

--- Result 3 ---
1 Linear Regression APNA COLLEGE Linear Regression Intuition & Logic Linear Regression is a supervised ML algorithm for regression problems. LR models the relationship between a dependent variable (output) and one or more independent variables (inputs) by fitting the best straight line (or plane/hyp
